In [ ]:
# Table 1 - recompute every exact P value from the published contingency counts
#
# The counts in the Table 1 sheet of Source Data are sufficient to reproduce
# all categorical tests; the age tests use the Supplementary Figure 4 sheet.
# No individual-level clinical record is required.
import pandas as pd
import numpy as np
from scipy import stats

SOURCE = "data/Source_Data.xlsx"   # published with the paper; see data/README.md
t1 = pd.read_excel(SOURCE, sheet_name="Table 1")
ages = pd.read_excel(SOURCE, sheet_name="Supplementary Figure 4")
t1.head(12)


In [ ]:
# split the sheet into one contingency table per variable block
blocks, current = {}, None
for _, r in t1.iterrows():
    var = str(r["Variable"]).strip()
    if var and var.lower() != "nan":
        current = var
        blocks[current] = []
    elif current and str(r["Category"]).strip().lower() not in ("", "nan"):
        a, b = r["Serotype 2 (n)"], r["Non-serotype 2 (n)"]
        if isinstance(a, (int, float)) and isinstance(b, (int, float)):
            blocks[current].append((str(r["Category"]), int(a), int(b)))

for k, v in blocks.items():
    if v:
        print(f"{k:30s} {len(v)} categories, "
              f"n={sum(x[1] for x in v)} vs {sum(x[2] for x in v)}")


In [ ]:
rng = np.random.default_rng(20261101)

def fisher_mc(tab, B=100_000):
    """Monte Carlo Fisher exact test for an r x 2 table."""
    obs = stats.chi2_contingency(tab)[0]
    rows, cols = tab.sum(1), tab.sum(0)
    labels, hits = np.repeat(np.arange(len(rows)), rows), 0
    for _ in range(B):
        perm, sim, i = rng.permutation(labels), np.zeros_like(tab), 0
        for j, cj in enumerate(cols):
            np.add.at(sim[:, j], perm[i:i + cj], 1)
            i += cj
        if stats.chi2_contingency(sim)[0] >= obs - 1e-9:
            hits += 1
    return (hits + 1) / (B + 1)

CHI_ONLY = {"Outcome"}
print(f"{'Variable':30s} {'chi2':>9s} {'df':>3s} {'exact P':>12s}")
print("-" * 58)
for name, rows_ in blocks.items():
    if not rows_:
        continue
    tab = np.array([[a, b] for _, a, b in rows_])
    tab = tab[tab.sum(1) > 0]
    chi2, pchi, dof, _ = stats.chi2_contingency(tab)
    p = pchi if name in CHI_ONLY else fisher_mc(tab, 20_000)
    print(f"{name:30s} {chi2:9.3f} {dof:3d} {p:12.4g}")


In [ ]:
s2    = ages.loc[ages["Group"] == "Serotype 2",      "Age (months)"].dropna()
other = ages.loc[ages["Group"] == "Other serotypes", "Age (months)"].dropna()

t, pt = stats.ttest_ind(s2, other, equal_var=False)
se = np.sqrt(s2.var(ddof=1) / len(s2) + other.var(ddof=1) / len(other))
dfw = se**4 / ((s2.var(ddof=1)/len(s2))**2/(len(s2)-1)
               + (other.var(ddof=1)/len(other))**2/(len(other)-1))
diff = other.mean() - s2.mean()
ci = stats.t.ppf(0.975, dfw) * se
u, pu = stats.mannwhitneyu(s2, other, alternative="two-sided")

print(f"Age, Welch t-test   t = {t:.3f}, df = {dfw:.1f}, P = {pt:.3g}")
print(f"                    mean difference {diff:.2f} months "
      f"(95% CI {diff-ci:.2f} to {diff+ci:.2f})")
print(f"Age, Wilcoxon       U = {u:.0f}, P = {pu:.3g}")
